In [1]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

## Tools

In [2]:
from langchain.tools import tool

In [3]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response


In [4]:
tool_duckduckgo_search.invoke("What is the capital of France?")

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_18912\2405146460.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


"1 day ago - Paris is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km2 (40.7 sq mi), as of January 2026, and a metropolitan population of 13.3 million (2023). Located on the river Seine in the centre of the Île-de-France region, it is the ... 2 weeks ago - This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944. 2 weeks ago - Its metropolitan area extends from ... and the North Sea. Its 18 integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi), with a total population estimated at over 69.1 million in 2026. Its capital, largest city and main cultural and economic centre is Paris, with a metropolitan population of over 13 million. Metropolitan France was settled ... 2 weeks ago - Le 10 juin 1940, face à l'arrivée ... français part à Tours puis continue sur Bordeaux. Le président de la République Albert Lebrun élit domicil

In [5]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    """Use this tool when you need to answer questions about general knowledge."""

    import wikipediaapi

    wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent='RajeshAIProject/1.0 (rajeshetl78456@xyz.com)'
    )

    page = wiki.page(query)

    if not page.exists():
        return f"No Wikipedia page found for: {query}"

    return page.summary


In [6]:
tool_wikipedia_search.invoke("Sachin Tendulkar")

'Sachin Ramesh Tendulkar ( ; Marathi: [sətɕin t̪eɳɖulkəɾ]; born 24 April 1973) is an Indian former international cricketer who captained the Indian national team. Tendulkar is widely regarded as one of the greatest cricketers in the history of cricket. He holds several world records, including being the all-time highest run-scorer in international cricket, receiving the most player of the match awards in international cricket, and being the only batsman to score 100 international centuries. Tendulkar was a Member of Parliament, Rajya Sabha by presidential nomination from 2012 to 2018.\nTendulkar took up cricket at the age of eleven, made his Test match debut on 15 November 1989 against Pakistan in Karachi at the age of sixteen, and went on to represent Mumbai domestically and India internationally for over 24 years. In 2002, halfway through his career, Wisden ranked him the second-greatest Test batsman of all time, behind Don Bradman, and the second-greatest ODI batsman of all time, be

In [7]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "Rajesh Venkatesan",
        "age": 34,
        "occupation": "Data Scientist"
    },
    {
        "name": "Lavanya Pugazhendi",
        "age": 31,
        "occupation": "Cyber Security Analyst"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."



In [8]:
tool_personal_info.invoke("Lavanya Pugazhendi")

'Lavanya Pugazhendi is 31 years old and works as a Cyber Security Analyst.'

In [9]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content
    

In [10]:
tool_rag.invoke("tell me about europe?")

C:\Users\Lavanya Rajesh\AppData\Local\Temp\ipykernel_18912\3480324300.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


'Europe is also home to animals of all sizes and shapes, \nand beautiful mountains and rivers. It is full of impressive \nstructures to see and delicious food to eat, as well! Most of the countries in Europe are on the \nmainland, which stretches from the North \nAtlantic Ocean in the west to the Ural \nMountains of Russia in the east, and from the frigid Arctic \nOcean north of Norway to the Mediterranean Sea down  \nsouth. These “mainland” countries are divided  \ninto Eastern, Northern, Southern, Central, \nand Western regions.\nYour \npal, T ricia, will finish your travels with a tour of these fine \ncountries. Mickey\nTricia\n8\nOne more thing! As you shall see, we have divided Europe into several regions (such as Central \nEurope).'

## Bind Tools

In [11]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_personal_info,
    tool_rag
]

In [12]:
llm_bind = llm.bind_tools(toolkit)

In [13]:
llm_bind.invoke("What's the age of Lavanya Pugazhendi?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 284, 'total_tokens': 378, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EP5t2l27JlCiXwSTdjbDhKEAmJOVL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0af6c-c867-7202-a09d-06859696f918-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'Lavanya Pugazhendi'}, 'id': 'call_22XoFp3tGAgW18b3edoJHtux', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 94, 'total_tokens': 378, 'input_token_detail

## ReAct Agent

In [14]:
from langchain.agents import create_agent


my_agent = create_agent(llm_bind, toolkit)

In [15]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of Lavanya Pugazhendi?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of Lavanya Pugazhendi?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='70525e69-12b6-405c-8a1c-983d209415b2'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 284, 'total_tokens': 314, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EP5tD1kMmgyhW2Eu8BY4oHQbab9cb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0af6c-efd3-7ef3-8e32-7e27e08e5bb6-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'Lavanya P

In [16]:
response = my_agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded?"}]}
)

## MEMORY

In [17]:
agent_memory = []

In [18]:
def invoke_agent_with_memory(query: str):

    response = my_agent.invoke(
        {"messages": [{"role": "user", "content": f"Answer this query: {query}. You can refer to the previous interactions : {str(agent_memory)}"}]}
    )
    agent_memory.extend(response['messages'])
    return response

In [19]:
# invoke_agent_with_memory("tell me about europe")

In [20]:
# invoke_agent_with_memory("Could you please summarize the previous interactions?")

In [21]:
# agent_memory

In [22]:
while True:
    query = input("Ask me anything: ")
    response = invoke_agent_with_memory(query)
    print(response)
    flag = input("Do you want to continue? (yes/no): ")
    if flag.lower() == "no":
        break

{'messages': [HumanMessage(content='Answer this query: tell me abour france. You can refer to the previous interactions : []', additional_kwargs={}, response_metadata={}, id='19d96d6b-2e38-4d93-9e9f-bebd69b906e4'), AIMessage(content='Sure — here’s a concise overview of France. (I corrected the typo “abour” to “about.” Let me know if you want more detail on any part.)\n\nBasic facts\n- Official name: French Republic (République française).\n- Capital: Paris.\n- Official language: French (regional languages such as Breton, Occitan, Corsican, Basque are also spoken).\n- Population: ~67 million (around mid-2020s).\n- Area: ~643,801 km² (metropolitan France).\n- Currency: Euro (EUR).\n- Time zone: Central European Time (CET/CEST in summer) for metropolitan France; overseas territories use other zones.\n\nGovernment & international role\n- System: semi-presidential republic (executive shared between president and prime minister).\n- France is a founding member of the EU, a Schengen state, a 